## GatorTron Training and Prediction

#### Import and Load Packages

In [1]:
import numpy as np
import pandas as pd
import transformers
import torch
import sys
import accelerate
import os
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, AutoModelForSequenceClassification,Trainer, TrainingArguments
from huggingface_hub import login
import sklearn

/home/sbalaj4/miniconda3/envs/hf-pytorch/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


#### MedSpacy Sectionizer and Sentence Filtering

Begin by filtering for specific sections in the .json, maintaining sections with the word "_retain."

For sections that are retained, filter to keep only sentences with any mention of metastasis or other words included in the "expanded metastasis terms" text file

In [ ]:
import re
import medspacy
from medspacy.section_detection import Sectionizer
from negspacy.negation import Negex
from spacy.pipeline import EntityRuler
from negspacy.termsets import termset

#load
nlp=medspacy.load(medspacy_enable='default')

#sentence splitting
if "sentencizer" not in nlp.pipe_names:
    nlp.add_pipe("sentencizer")

#filter by section based on section names in the .json
sectionizer = Sectionizer(
    nlp,
    rules="/home/sbalaj4/.config/sections.json",
    name="custom_sectionizer"
)

#keywords for sentence filtering
def load_metastasis_words(filepath):
    keywords = []
    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            parts = [w.strip().lower() for w in line.split(",") if w.strip()]
            keywords.extend(parts)
    return keywords

filepath = "/home/sbalaj4/expanded_metastasis_terms copy.txt"
METASTASES_WORDS = load_metastasis_words(filepath)

#negation detection
ts=termset("en_clinical")

#specify negation patterns
patterns=ts.get_patterns()
patterns["preceding_negations"].extend(["no evidence of", "absence of","no presence", "without presence of"])

#remove these negations
patterns["preceding_negations"]=[p for p in patterns["preceding_negations"] if p !="denies"]

#add to nlp pipeline
nlp.add_pipe("negex", config={"neg_termset":patterns,"ent_types":["METS"],"extension_name":"negex","chunk_prefix":["no"]},last=True)

ruler = nlp.add_pipe("entity_ruler", before="negex")
patterns = [{"label": "METS", "pattern": w} for w in METASTASES_WORDS]
ruler.add_patterns(patterns)


#clean test for symbols
def clean_text(text):
    text = str(text) if text is not None else ""
    text = re.sub(r"^\s*[-•]?\s*\d+[\.\)]?\s*", "", text)
    text = re.sub(r"\d+", "", text)
    text = re.sub(r"[^A-Za-z0-9\s]", "", text)
    text = re.sub(r"\s{2,}", " ", text)
    return text.strip()


def positive_sent(text, keywords, nlp):
    doc = nlp(text)
    found_positive = False
    for ent in doc.ents:
        if ent.label_ == "METS":  
            if not getattr(ent._, "negex", False):  
                found_positive = True
    return found_positive

#preprocess 
def preprocess_note(text, nlp, sectionizer, METASTASES_WORDS):
    doc = nlp(text)
    sectionizer(doc)
    revised_note = []
    
    contains_mets = any(positive_sent(sent.text, METASTASES_WORDS, nlp) for sent in doc.sents)
    
    if contains_mets:
        for sent in doc.sents:
            if positive_sent(sent.text, METASTASES_WORDS, nlp): #keep only sentences with positive mentions of the keyword, not negated
                revised_note.append(clean_text(sent.text))
    else:
        for i, title in enumerate(doc._.section_titles): #keep sections specified in the .json if no keywords are found in the note.
            categories = doc._.section_categories[i]
            body_text = str(doc._.section_bodies[i])
            if categories is None:
                categories = []
            elif isinstance(categories, str):
                categories = [categories]
            categories = [c.lower().strip() for c in categories if c]
            if any("_retain" in c for c in categories) and body_text.strip():
                revised_note.append(f"{title}\n\t{clean_text(body_text)}")

    return "\n".join([s for s in revised_note if s])


#### Apply the Sectionizing and Sentence filtering to the MIMIC notes

In [ ]:
input_text=pd.read_csv("/home/sbalaj4/mets_ext_sample_data_9.23.csv")

#specify column with the notes - 'TEXT' column
filtered_set = input_text[input_text['TEXT'].notna()].copy()
filtered_set["TEXT"] = filtered_set["TEXT"].astype(str)

#detect mets keywords
filtered_set["contains_mets"] = filtered_set["TEXT"].apply(
    lambda x: positive_sent(sent.text, METASTASES_WORDS, nlp)
)

filtered_set["revised_note"] = filtered_set.apply(
    lambda row: preprocess_note(row["TEXT"], nlp, sectionizer, METASTASES_WORDS),
    axis=1
)

#remove any notes that are empty after filtering
filtered_set = filtered_set[filtered_set['revised_note'].str.strip() != '']
print(f"Total notes after preprocessing: {len(filtered_set)}")

#save notes
filtered_set.to_csv("/home/sbalaj4/filtered_mets_notes_9.29.csv", index=False)

Total notes after preprocessing: 213


#### Train GatorTron on MIMIC Notes

In [4]:
import pandas as pd
import numpy as np
import torch
from datasets import Dataset
from sklearn.model_selection import train_test_split
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)

#load data
df = pd.read_csv("/home/sbalaj4/filtered_mets_notes_ALTERED_10.3.csv")

#convert column with clinical notes to a string
df["revised_note"] = df["revised_note"].astype(str)

#load gatortron
model_id = "UFNLP/gatortron-base"
tokenizer = AutoTokenizer.from_pretrained(model_id)

def tokenize(batch):
    return tokenizer(
        batch["revised_note"],
        truncation=True,
        padding="max_length",
        max_length=512,
    )

#classify presence of metastasis
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42) #train test split

#convert to huggingface approved dataset
train_ds_presence = Dataset.from_pandas(train_df[["revised_note", "presence"]])
val_ds_presence   = Dataset.from_pandas(val_df[["revised_note", "presence"]])

train_ds_presence = train_ds_presence.map(tokenize, batched=True)
val_ds_presence   = val_ds_presence.map(tokenize, batched=True)

train_ds_presence = train_ds_presence.rename_column("presence", "labels")
val_ds_presence   = val_ds_presence.rename_column("presence", "labels")

cols = ["input_ids", "attention_mask", "labels"]
train_ds_presence.set_format(type="torch", columns=cols)
val_ds_presence.set_format(type="torch", columns=cols)

model_presence = AutoModelForSequenceClassification.from_pretrained(model_id, num_labels=2)

#train model and save it locally
training_args_presence = TrainingArguments(
    output_dir="/home/sbalaj4/presence_cls",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=5,
    weight_decay=0.01,
    fp16=True,
    logging_dir="/home/sbalaj4/logs_presence")

trainer_presence = Trainer(
    model=model_presence,
    args=training_args_presence,
    train_dataset=train_ds_presence,
    eval_dataset=val_ds_presence,
    tokenizer=tokenizer)

trainer_presence.train()
trainer_presence.save_model()

#classify location of metastasis (for notes where presence = 1)

#names of location columns
location_cols = ["lung", "liver", "abdominal", "CNS", "lymph_node", "bone", "other"]

#only applies to notes where presence = 1
df_loc = df[df["presence"] == 1].copy()

#location columns should be numeric (0/1)
for col in location_cols:
    df_loc[col] = pd.to_numeric(df_loc[col], errors="coerce").fillna(0).astype(float)

Y = df_loc[location_cols].values.astype("float32")

train_df_loc, val_df_loc, y_train, y_val = train_test_split(df_loc, Y, test_size=0.2, random_state=42)

train_ds_loc = Dataset.from_pandas(pd.DataFrame({"revised_note": train_df_loc["revised_note"], "labels": list(y_train)}))
val_ds_loc = Dataset.from_pandas(pd.DataFrame({"revised_note": val_df_loc["revised_note"], "labels": list(y_val)}))

train_ds_loc = train_ds_loc.map(tokenize, batched=True)
val_ds_loc   = val_ds_loc.map(tokenize, batched=True)

def cast_labels(example):
    example["labels"] = torch.tensor(example["labels"], dtype=torch.float)
    return example

train_ds_loc = train_ds_loc.map(cast_labels)
val_ds_loc   = val_ds_loc.map(cast_labels)

cols = ["input_ids", "attention_mask", "labels"]
train_ds_loc.set_format(type="torch", columns=cols)
val_ds_loc.set_format(type="torch", columns=cols)

#use multilabel classification, since a single note can have multiple different metastasis labels
num_labels = len(location_cols)
model_location = AutoModelForSequenceClassification.from_pretrained(
    model_id,
    num_labels=num_labels,
    problem_type="multi_label_classification")

#train model and save locally
training_args_location = TrainingArguments(
    output_dir="/home/sbalaj4/location_cls",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=5,
    weight_decay=0.01,
    fp16=True,
    logging_dir="/home/sbalaj4/logs_location")

trainer_location = Trainer(
    model=model_location,
    args=training_args_location,
    train_dataset=train_ds_loc,
    eval_dataset=val_ds_loc,
    tokenizer=tokenizer)

trainer_location.train()
trainer_location.save_model()

#predictions on the full dataset
ds_presence_full = Dataset.from_pandas(df[["revised_note"]])
ds_presence_full = ds_presence_full.map(tokenize, batched=True)
ds_presence_full.set_format(type="torch", columns=["input_ids", "attention_mask"])

raw_preds_presence = trainer_presence.predict(ds_presence_full)
logits_presence = raw_preds_presence.predictions
df["pred_presence"] = np.argmax(logits_presence, axis=-1)

for col in location_cols:
    df[f"pred_{col}"] = 0

#predict location where presence = 1
df_pred_loc = df[df["pred_presence"] == 1].copy()

if not df_pred_loc.empty:
    ds_loc_full = Dataset.from_pandas(df_pred_loc[["revised_note"]])
    ds_loc_full = ds_loc_full.map(tokenize, batched=True)
    ds_loc_full.set_format(type="torch", columns=["input_ids", "attention_mask"])

    raw_preds_loc = trainer_location.predict(ds_loc_full)
    logits_loc = raw_preds_loc.predictions
    probs_loc = 1 / (1 + np.exp(-logits_loc))  # sigmoid
    preds_loc = (probs_loc > 0.5).astype(int)

    preds_df = pd.DataFrame(preds_loc, columns=[f"pred_{c}" for c in location_cols])
    df.loc[df["pred_presence"] == 1, preds_df.columns] = preds_df.values

    #mark as "other" if no specific metastasis site is mentioned but there is metastasis.
    mask_no_site = (preds_loc[:, :-1].sum(axis=1) == 0)
    df.loc[df["pred_presence"] == 1, "pred_other"] = (df.loc[df["pred_presence"] == 1, "pred_other"] | mask_no_site)

#save
df.to_csv("/home/sbalaj4/mimic_mets_predictions_full.csv", index=False)

Map: 100%|██████████| 43/43 [00:00<00:00, 1500.02 examples/s]
Some weights of MegatronBertForSequenceClassification were not initialized from the model checkpoint at UFNLP/gatortron-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Step,Training Loss


Map: 100%|██████████| 12/12 [00:00<00:00, 4055.08 examples/s]
Some weights of MegatronBertForSequenceClassification were not initialized from the model checkpoint at UFNLP/gatortron-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Step,Training Loss


Map: 100%|██████████| 213/213 [00:00<00:00, 1702.14 examples/s]


Map: 100%|██████████| 56/56 [00:00<00:00, 3639.14 examples/s]


#### Apply tuned models to the real clinical notes

##### Begin by filtering notes using the medspacy code above and then applying it to the clinical notes below

In [9]:
input_text=pd.read_csv("/labs/bozkurtlab/metastasis-data/Radiology_SB_NOTES_22pts_061725.csv")

#specify column with the notes - 'TEXT' column
filtered_set = input_text[input_text['EVENT_DOC_TXT0'].notna()].copy()
filtered_set["EVENT_DOC_TXT0"] = filtered_set["EVENT_DOC_TXT0"].astype(str)

#detect mets keywords
filtered_set["contains_mets"] = filtered_set["EVENT_DOC_TXT0"].apply(lambda x: positive_sent(x, METASTASES_WORDS, nlp))
filtered_set["revised_note"] = filtered_set.apply(lambda row: preprocess_note(row["EVENT_DOC_TXT0"], nlp, sectionizer, METASTASES_WORDS),axis=1)

#remove any notes that are empty after filtering
filtered_set = filtered_set[filtered_set['revised_note'].str.strip() != '']
print(f"Total notes after preprocessing: {len(filtered_set)}")

#save notes
filtered_set.to_csv("/home/sbalaj4/filtered_REAL_notes_10.1.csv", index=False)

Total notes after preprocessing: 643


#### Use the models saved locally to predict presence + location of metastasis in the real clinical notes

In [10]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer
from datasets import Dataset
import pandas as pd
import numpy as np

#load model for mets presence 
presence_model_dir = "/home/sbalaj4/presence_cls"

#load model for mets location
location_model_dir = "/home/sbalaj4/location_cls"

tokenizer = AutoTokenizer.from_pretrained(presence_model_dir)
model_presence = AutoModelForSequenceClassification.from_pretrained(presence_model_dir)
model_location = AutoModelForSequenceClassification.from_pretrained(location_model_dir)

trainer_presence = Trainer(model=model_presence, tokenizer=tokenizer)
trainer_location = Trainer(model=model_location, tokenizer=tokenizer)

#load real clinical notes
df_new = pd.read_csv("/home/sbalaj4/filtered_REAL_notes_10.1.csv")

#specify column with the filtered notes
df_new["revised_note"] = df_new["revised_note"].fillna("").astype(str)

def tokenize(batch):
    return tokenizer(batch["revised_note"], truncation=True, padding="max_length", max_length=512)

#predict presence of metastasis

ds_presence = Dataset.from_pandas(df_new[["revised_note"]])
ds_presence = ds_presence.map(tokenize, batched=True, batch_size=16)
ds_presence.set_format(type="torch", columns=["input_ids", "attention_mask"])

raw_preds_presence = trainer_presence.predict(ds_presence)
logits_presence = raw_preds_presence.predictions
df_new["pred_presence"] = np.argmax(logits_presence, axis=-1)

#predict location of metastasis
location_cols = ["lung", "liver", "abdominal", "CNS", "lymph_node", "bone", "other"]

# initialize all as 0 first
for col in location_cols:
    df_new[f"pred_{col}"] = 0

#make sure location is being predicted for notes that have metastasis only

df_with_mets = df_new[df_new["pred_presence"] == 1].copy()
if not df_with_mets.empty:
    ds_loc = Dataset.from_pandas(df_with_mets[["revised_note"]])
    ds_loc = ds_loc.map(tokenize, batched=True, batch_size=16)
    ds_loc.set_format(type="torch", columns=["input_ids", "attention_mask"])

    raw_preds_loc = trainer_location.predict(ds_loc)
    logits_loc = raw_preds_loc.predictions
    probs_loc = 1 / (1 + np.exp(-logits_loc))  # sigmoid
    preds_loc = (probs_loc > 0.5).astype(int)

    preds_df = pd.DataFrame(preds_loc, columns=[f"pred_{c}" for c in location_cols])
    df_new.loc[df_new["pred_presence"] == 1, preds_df.columns] = preds_df.values

    mask_no_site = (preds_loc.sum(axis=1) == 0)
    df_new.loc[df_new["pred_presence"] == 1, "pred_other"] = df_new.loc[df_new["pred_presence"] == 1, "pred_other"] | mask_no_site

#save
df_new.to_csv("/home/sbalaj4/clinical_notes_mets_prediction.csv", index=False)

Map: 100%|██████████| 643/643 [00:00<00:00, 2818.86 examples/s]


Map: 100%|██████████| 291/291 [00:00<00:00, 4412.46 examples/s]
